# Pretraining and Evaluation Notebook for TELUGU
This notebook implements: 
1. Dataset streaming from jsonl files.
2. Training loop with checkpoint resume and learning rate scheduling.
3. Validation tracking (cross-entropy, perplexity, and Bits-Per-Byte).
4. Autoregressive sample generation and metrics evaluation (BLEU-4, ROUGE-L, chrF, Distinct-1/2, repetition rate).
5. Attention analysis (heatmaps, entropy, and mean distance).
6. Automatic plotting and saving to `/figures` directory.

In [ ]:
import os
import sys
import json
import math
import yaml
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import urllib.request
import seaborn as sns
from tokenizers import Tokenizer
from collections import Counter
from torch.utils.data import IterableDataset, DataLoader
from datasets import load_dataset

lang = "telugu"

# Dynamically download and register Indic font for matplotlib to prevent tofu/rectangular boxes in plots
font_config = {
    'telugu': {
        'url': 'https://github.com/notofonts/noto-fonts/raw/main/unhinted/ttf/NotoSansTelugu/NotoSansTelugu-Regular.ttf',
        'filename': 'NotoSansTelugu-Regular.ttf'
    },
    'nepali': {
        'url': 'https://github.com/notofonts/noto-fonts/raw/main/unhinted/ttf/NotoSansDevanagari/NotoSansDevanagari-Regular.ttf',
        'filename': 'NotoSansDevanagari-Regular.ttf'
    }
}
try:
    cfg = font_config[lang]
    if not os.path.exists(cfg['filename']):
        print(f"[INFO] Downloading Noto Sans font for {lang}... ")
        urllib.request.urlretrieve(cfg['url'], cfg['filename'])
    fm.fontManager.addfont(cfg['filename'])
    # Store FontProperties for EXPLICIT per-element use only.
    # DO NOT insert into rcParams font.sans-serif: Noto Sans Telugu has no Latin glyphs,
    # so setting it globally makes every English letter a rectangle box.
    _indic_font_prop = fm.FontProperties(fname=cfg['filename'])
    plt.rcParams['axes.unicode_minus'] = False
    print(f"[INFO] Successfully registered Indic font: {_indic_font_prop.get_name()}")
except Exception as e:
    _indic_font_prop = None
    print(f"[WARNING] Failed to download/register Noto Sans font: {e}")

def _reset_to_latin():
    """Reset matplotlib to use DejaVu Sans (full Latin coverage) for all NEW figures.
    Call this at the top of every plot function to prevent the Indic font
    from leaking into titles, axis labels, ticks, and legends.
    """
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Liberation Sans', 'Arial', 'Helvetica']

# Add project root to sys.path to enable package imports
sys.path.append(str(Path("../../")))
from transformer.arch import Transformer

In [ ]:
# Load model configurations
config_path = "config.yaml" if os.path.exists("config.yaml") else "../config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)
print("Model Configuration:")
for k, v in config.items():
    print(f"  {k}: {v}")

In [ ]:
class TeluguDataset(IterableDataset):
    """
    Dataset loader that streams raw text from a Hugging Face Dataset split,
    tokenizes on-the-fly, and chops into contiguous blocks of max_seq_len.
    """
    def __init__(self, hf_dataset, tokenizer, max_seq_len):
        super().__init__()
        self.hf_dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len

    def __iter__(self):
        current_tokens = []
        for row in self.hf_dataset:
            text = row.get("text", "")
            if not text or not text.strip():
                continue
            tokens = self.tokenizer.encode(text).ids
            current_tokens.extend(tokens)
            
            while len(current_tokens) >= self.max_seq_len + 1:
                segment = current_tokens[:self.max_seq_len + 1]
                current_tokens = current_tokens[self.max_seq_len:]
                yield torch.tensor(segment[:-1], dtype=torch.long), torch.tensor(segment[1:], dtype=torch.long)


## 1. Load Dataset from Hugging Face Hub

We retrieve the train, validation, and test splits directly from the Hugging Face dataset repository using Hugging Face's `load_dataset` function.

In [ ]:
# Load the dataset splits from Hugging Face Hub
hf_dataset = load_dataset(
    "Nikhil8076/telugu-monolingual-dataset",
    data_files={
        "train": "cleaned/telugu_train.jsonl",
        "validation": "cleaned/telugu_val.jsonl",
        "test": "cleaned/telugu_test.jsonl"
    }
)
print("Loaded Dataset splits:", hf_dataset)

In [ ]:
def save_checkpoint(model, optimizer, scheduler, step, train_losses, val_losses, filename='checkpoint.pth', checkpoint_dir='.'):
    os.makedirs(checkpoint_dir, exist_ok=True)
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'step': step,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'config': config
    }
    torch.save(checkpoint, os.path.join(checkpoint_dir, filename))
    print(f"[INFO] Checkpoint saved successfully: {filename} at step {step}.")

def load_checkpoint(model, optimizer, scheduler, checkpoint_path="checkpoint.pth"):
    if os.path.exists(checkpoint_path):
        print(f"[INFO] Resuming training from existing checkpoint: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=next(model.parameters()).device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        return checkpoint['step'], checkpoint.get('train_losses', []), checkpoint.get('val_losses', [])
    print("[INFO] No checkpoint found. Initializing model training from scratch.")
    return 0, [], []

In [ ]:
def calculate_bpb(loss_val, num_tokens, num_chars):
    """
    Bits-Per-Byte: normalises cross-entropy loss by character count so
    models with different tokenizer fertilities are comparable.
    Formula: (loss * num_tokens) / (num_chars * ln 2)
    """
    if num_chars == 0:
        return float('inf')
    return (loss_val * num_tokens) / (num_chars * math.log(2))


In [ ]:
def evaluate_loss(model, dataset_split, batch_size=16):
    """Runs validation evaluation → (avg_loss, perplexity, BPB)."""
    model.eval()
    dataset = TeluguDataset(dataset_split, tokenizer, config['max_seq_len'])
    loader = DataLoader(dataset, batch_size=batch_size)
    total_loss, num_batches, num_tokens = 0.0, 0, 0
    num_chars = sum(len(row.get("text", "")) for row in dataset_split)
    with torch.no_grad():
        for x, y in loader:
            x = x.to(next(model.parameters()).device)
            y = y.to(next(model.parameters()).device)
            _, loss, _ = model(x, y)
            total_loss += loss.item()
            num_batches += 1
            num_tokens += y.numel()
    if num_batches == 0:
        return float('inf'), float('inf'), float('inf')
    avg_loss = total_loss / num_batches
    perplexity = math.exp(avg_loss)
    bpb = calculate_bpb(avg_loss, num_tokens, num_chars)
    return avg_loss, perplexity, bpb


In [ ]:
# Setup device GPU/MPS acceleration
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"Using device: {device}")

# Load tokenizer (needed for on-the-fly tokenization in the dataset class)
tokenizer = Tokenizer.from_file(f"../tokenizer/telugu_tokenizer.json")

# Pretraining parameters
max_steps = 20000
batch_size = 16
grad_accum_steps = 4

# Helper to construct the cosine annealing scheduler with linear warmup
def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps, min_lr_ratio=0.1):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))
        return min_lr_ratio + (1.0 - min_lr_ratio) * cosine_decay
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Initialize model
model = Transformer(**config).to(device)

# Exclude 1D tensors (like RMSNorm parameters) and biases from weight decay
decay_params, no_decay_params = [], []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if param.ndim < 2 or "bias" in name:
        no_decay_params.append(param)
    else:
        decay_params.append(param)
optim_groups = [
    {"params": decay_params, "weight_decay": 0.01},
    {"params": no_decay_params, "weight_decay": 0.0}
]
optimizer = torch.optim.AdamW(optim_groups, lr=6e-4)

# Initialize the Cosine Annealing with Warmup scheduler
total_opt_steps = max_steps // grad_accum_steps
warmup_steps = int(total_opt_steps * 0.1)  # 10% of total training steps
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_opt_steps,
    min_lr_ratio=0.1
)

# Load checkpoint
start_step, train_losses, val_losses = load_checkpoint(model, optimizer, scheduler)
# Track which optimizer-update steps had validation computed (for x-axis alignment)
val_steps_recorded = []

print(f"Pretraining TELUGU model...")

def run_pretraining(max_steps=max_steps, batch_size=batch_size, grad_accum_steps=grad_accum_steps):
    global val_steps_recorded
    step = start_step
    if step >= max_steps:
        print("[INFO] Already pretrained to target steps.")
        return

    dataset = TeluguDataset(hf_dataset['train'], tokenizer, config['max_seq_len'])
    loader = DataLoader(dataset, batch_size=batch_size)

    best_val_loss = min(val_losses) if len(val_losses) > 0 else float('inf')
    running_loss = 0.0
    optimizer.zero_grad()
    opt_step_count = step // grad_accum_steps  # how many optimizer updates so far

    for x, y in loader:
        if step >= max_steps:
            break

        model.train()
        x, y = x.to(device), y.to(device)
        logits, loss, _ = model(x, y)
        loss = loss / grad_accum_steps
        loss.backward()
        running_loss += loss.item() * grad_accum_steps

        if (step + 1) % grad_accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            opt_step_count += 1

            train_losses.append(running_loss / grad_accum_steps)
            running_loss = 0.0

            current_lr = scheduler.get_last_lr()[0]
            print(f"Step {step} | Loss: {train_losses[-1]:.4f} | LR: {current_lr:.2e}")

            # Validation cycle every 500 optimizer updates
            if opt_step_count % 500 == 0:
                val_loss, ppl, bpb = evaluate_loss(model, hf_dataset['validation'], batch_size)
                val_losses.append(val_loss)
                val_steps_recorded.append(opt_step_count)  # record where in opt-step space this val was
                print(f"Step {step} | Opt Step {opt_step_count} | Val Loss: {val_loss:.4f} | PPL: {ppl:.2f} | BPB: {bpb:.4f}")
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    save_checkpoint(model, optimizer, scheduler, step, train_losses, val_losses, filename='best_checkpoint.pth')
                    print(f"New best validation loss: {best_val_loss:.4f} → saved best_checkpoint.pth")

            # Periodic checkpoint every 1250 optimizer updates
            if opt_step_count % 1250 == 0:
                save_checkpoint(model, optimizer, scheduler, step, train_losses, val_losses, filename='checkpoint.pth')

        # Explicit milestone checkpoint at step 10,000
        if step + 1 == 10000:
            save_checkpoint(model, optimizer, scheduler, step, train_losses, val_losses, filename='checkpoint_10k.pth')
            print("[INFO] Mid-training milestone checkpoint saved: checkpoint_10k.pth")

        step += 1

    # Save final training state at the end
    save_checkpoint(model, optimizer, scheduler, step, train_losses, val_losses, filename='checkpoint.pth')

run_pretraining(max_steps=max_steps, batch_size=batch_size, grad_accum_steps=grad_accum_steps)


## 2. Loss Curve Visualization

Plots the training and validation cross-entropy losses over training steps to diagnose optimization progress, convergence speed, and potential overfitting.

In [ ]:
def save_loss_curve(train_losses, val_losses, val_steps_recorded=None, figures_dir="figures"):
    """Generates and saves the training/validation loss curve."""
    _reset_to_latin()  # ensure all text uses Latin font (no Indic glyph boxes)
    os.makedirs(figures_dir, exist_ok=True)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(range(len(train_losses)), train_losses, label='Train Loss', color='royalblue', linewidth=1.5)
    if len(val_losses) > 0:
        # Use recorded optimizer-step positions if available, else fall back to linspace
        if val_steps_recorded and len(val_steps_recorded) == len(val_losses):
            xs = val_steps_recorded
        else:
            xs = list(np.linspace(0, len(train_losses) - 1, len(val_losses)).astype(int))
        ax.plot(xs, val_losses, label='Validation Loss', color='darkorange', marker='o', linewidth=1.5)
    ax.set_xlabel('Optimizer Steps', fontfamily='DejaVu Sans')
    ax.set_ylabel('Cross-Entropy Loss', fontfamily='DejaVu Sans')
    ax.set_title('Pretraining Loss Curve (TELUGU)', fontfamily='DejaVu Sans')
    ax.legend(prop={'family': 'DejaVu Sans'})
    ax.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plot_path = os.path.join(figures_dir, 'loss_curve.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"[INFO] Loss curve saved to {plot_path}")

save_loss_curve(train_losses, val_losses, val_steps_recorded)


## 3. Intrinsic Language Modeling Metrics

Three standard intrinsic metrics are computed on the held-out validation split:

### A. Cross-Entropy Loss
$$L = -\frac{1}{T} \sum_{t=1}^T \log P(x_t \mid x_{<t})$$

### B. Perplexity (PPL)
$$\text{PPL} = e^L$$

### C. Bits-Per-Byte (BPB)
BPB normalises the loss by raw character count rather than token count. This makes two models with different tokenizer fertilities directly comparable even if one language uses more bytes per token:
$$\text{BPB} = \frac{L \cdot N_{\text{tokens}}}{N_{\text{chars}} \cdot \ln(2)}$$

In [ ]:
# Final intrinsic evaluation on validation set
val_loss, ppl, bpb = evaluate_loss(model, hf_dataset['validation'])
print(f"=== Intrinsic Metrics ({lang.upper()}) ===")
print(f"  Validation Cross-Entropy Loss : {val_loss:.4f}")
print(f"  Perplexity (PPL)              : {ppl:.2f}")
print(f"  Bits-Per-Byte (BPB)           : {bpb:.4f}")


## 4. Text Generation

Autoregressive next-token prediction with greedy decoding (temperature = 0) and temperature-scaled stochastic sampling at T = 0.5, 1.0, and 1.5.

For each generated sample we also compute:
- **BLEU-4**: n-gram precision vs. a reference continuation from the held-out set.
- **chrF**: character n-gram F-score (robust for morphologically rich scripts).
- **ROUGE-L**: longest common subsequence recall.

> **Note on metric validity for open-ended generation:**
> BLEU and ROUGE-L are designed for deterministic tasks (machine translation, summarisation) where a single correct reference exists. For open-ended generation they penalise valid but divergent continuations and should be interpreted with caution. chrF is somewhat more forgiving because it operates at the character level, making it more suitable for agglutinative scripts like Telugu and Nepali where surface forms vary greatly. The most informative diagnostics for open-ended LM quality are therefore BPB (how well the model compresses the test distribution) and the fluency/diversity metrics (Distinct-1/2, Repetition Rate).

In [ ]:
def generate_text(model, tokenizer, prompt, max_new_tokens=60, temperature=1.0):
    """Generates text continuation from a prompt using greedy or temperature sampling."""
    model.eval()
    tokens = tokenizer.encode(prompt).ids
    input_ids = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)
    for _ in range(max_new_tokens):
        cond_input = input_ids[:, -config['max_seq_len']:]
        with torch.no_grad():
            logits, _, _ = model(cond_input)
        next_logits = logits[:, -1, :]
        if temperature == 0.0:
            next_token = torch.argmax(next_logits, dim=-1, keepdim=True)
        else:
            next_logits = next_logits / temperature
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat((input_ids, next_token), dim=-1)
    return tokenizer.decode(input_ids[0].tolist())


## 5. BLEU-4
$$\text{BLEU-4} = \text{BP} \cdot \exp\left( \sum_{n=1}^4 0.25 \cdot \ln(p_n) \right), \quad \text{BP} = \min\!\left(1,\, e^{1 - r/c}\right)$$

In [ ]:
def compute_bleu_4(reference_text, candidate_text):
    """Corpus-level BLEU-4 (sentence-level approximation) implemented from scratch."""
    ref_words = reference_text.split()
    cand_words = candidate_text.split()
    if not ref_words or not cand_words:
        return 0.0
    p_ns = []
    for n in range(1, 5):
        ref_g = Counter(tuple(ref_words[i:i+n]) for i in range(len(ref_words) - n + 1))
        cand_g = Counter(tuple(cand_words[i:i+n]) for i in range(len(cand_words) - n + 1))
        if not ref_g or not cand_g:
            p_ns.append(1e-9)
            continue
        shared = sum(min(cand_g[g], ref_g[g]) for g in cand_g)
        total_cand = sum(cand_g.values())
        p_ns.append(shared / total_cand if total_cand > 0 else 1e-9)
    p_ns = [max(p, 1e-9) for p in p_ns]
    c_len, r_len = len(cand_words), len(ref_words)
    bp = 1.0 if c_len >= r_len else math.exp(1 - r_len / c_len) if c_len > 0 else 0.0
    return bp * math.exp(sum(0.25 * math.log(p) for p in p_ns))


## 6. chrF (Character n-gram F-score)
$$\text{chrF} = (1+\beta^2) \frac{P \cdot R}{\beta^2 P + R}, \quad \beta = 3$$

In [ ]:
def compute_chrf(reference_text, candidate_text, beta=3.0, max_n=6):
    """chrF with beta=3 (recall-heavy) implemented from scratch."""
    def char_ngrams(text, n):
        s = text.replace(" ", "")
        return Counter(s[i:i+n] for i in range(len(s) - n + 1))
    precisions, recalls = [], []
    for n in range(1, max_n + 1):
        ref_c = char_ngrams(reference_text, n)
        cand_c = char_ngrams(candidate_text, n)
        if not ref_c or not cand_c:
            continue
        shared = sum(min(cand_c[g], ref_c[g]) for g in cand_c)
        precisions.append(shared / sum(cand_c.values()))
        recalls.append(shared / sum(ref_c.values()))
    if not precisions:
        return 0.0
    P = sum(precisions) / len(precisions)
    R = sum(recalls) / len(recalls)
    denom = beta**2 * P + R
    return (1 + beta**2) * P * R / denom if denom > 0 else 0.0


## 7. ROUGE-L (Longest Common Subsequence)
$$\text{ROUGE-L} = \frac{2 \cdot R_{\text{LCS}} \cdot P_{\text{LCS}}}{R_{\text{LCS}} + P_{\text{LCS}}}$$

In [ ]:
def compute_rouge_l(reference_text, candidate_text):
    """Sentence ROUGE-L based on character-level LCS (better for Indic scripts)."""
    ref = list(reference_text)   # character-level for Indic scripts
    cand = list(candidate_text)
    m, n = len(ref), len(cand)
    if m == 0 or n == 0:
        return 0.0
    # DP LCS
    prev = [0] * (n + 1)
    for i in range(m):
        curr = [0] * (n + 1)
        for j in range(n):
            if ref[i] == cand[j]:
                curr[j+1] = prev[j] + 1
            else:
                curr[j+1] = max(prev[j+1], curr[j])
        prev = curr
    lcs_len = prev[n]
    r_lcs = lcs_len / m
    p_lcs = lcs_len / n
    denom = r_lcs + p_lcs
    return (2 * r_lcs * p_lcs) / denom if denom > 0 else 0.0


## 8. Distinct-1 / Distinct-2 and Repetition Rate

Distinct-N measures vocabulary diversity by computing the fraction of unique n-grams in the generated output.
Repetition Rate measures the fraction of repeated 4-grams (loop collapse diagnostic).

In [ ]:
def compute_distinct_n(tokens, n=1):
    """Distinct-N: fraction of unique n-grams."""
    if len(tokens) < n:
        return 0.0
    ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]
    return len(set(ngrams)) / len(ngrams)

def compute_repetition_rate(tokens, n=4):
    """Repetition Rate: fraction of repeating 4-grams."""
    if len(tokens) < n:
        return 0.0
    ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]
    return (len(ngrams) - len(set(ngrams))) / len(ngrams)


## 9. Generation Quality Evaluation

For each temperature (greedy / 0.5 / 1.0 / 1.5) we generate a continuation and score it against the gold reference continuation from the test set.

In [ ]:
prompt = "భారతదేశం నా మాతృభూమి"
max_new_tokens = 60

# Pull a clean reference from the test set (skip dirty/OCR/short rows)
import unicodedata
reference_text = ""
for row in hf_dataset['test']:
    t = row.get('text', '').strip()
    if len(t) < 50:
        continue  # too short
    alpha_frac = sum(1 for ch in t if unicodedata.category(ch)[0] in ('L', 'M')) / len(t)
    if alpha_frac < 0.75:
        continue  # too many digits/punctuation/special chars (likely OCR noise)
    reference_text = t
    break
if not reference_text:
    reference_text = "N/A"  # fallback if all rows are dirty

temperatures = [0.0, 0.5, 1.0, 1.5]
temp_labels  = ['Greedy (T=0)', 'T=0.5', 'T=1.0', 'T=1.5']

print(f"=== Generation Quality Evaluation (TELUGU) ===")
print(f"Prompt       : {prompt}")
print(f"Reference    : {reference_text[:120]}...\n")

gen_results = {}
for temp, label in zip(temperatures, temp_labels):
    gen_out = generate_text(model, tokenizer, prompt, max_new_tokens=max_new_tokens, temperature=temp)
    gen_tokens = tokenizer.encode(gen_out).ids
    bleu   = compute_bleu_4(reference_text, gen_out)
    chrf   = compute_chrf(reference_text, gen_out)
    rougel = compute_rouge_l(reference_text, gen_out)
    dist1  = compute_distinct_n(gen_tokens, 1)
    dist2  = compute_distinct_n(gen_tokens, 2)
    rep    = compute_repetition_rate(gen_tokens, 4)
    gen_results[label] = dict(gen=gen_out, bleu=bleu, chrf=chrf, rougel=rougel,
                               dist1=dist1, dist2=dist2, rep=rep)
    print(f"[{label}]")
    print(f"  Generated  : {gen_out[:120]}")
    print(f"  BLEU-4     : {bleu:.4f}")
    print(f"  chrF       : {chrf:.4f}")
    print(f"  ROUGE-L    : {rougel:.4f}")
    print(f"  Distinct-1 : {dist1:.4f} | Distinct-2: {dist2:.4f} | Rep.Rate: {rep:.4f}")
    print()


## 10. Attention Matrix Analysis

For each transformer layer we extract the query-key attention matrix $A_{h,i,j}$ and compute two head-level statistics:

**A. Shannon Attention Entropy (bits)**
$$H(A_h) = -\frac{1}{T} \sum_{i=1}^T \sum_{j=1}^T A_{h,i,j} \log_2(A_{h,i,j} + \varepsilon)$$
High entropy → diffuse / global attention. Low entropy → sharp / local attention.

**B. Mean Attention Distance**
$$D(A_h) = \frac{1}{T} \sum_{i=1}^T \sum_{j=1}^T A_{h,i,j} \cdot |i - j|$$
Short distance → local / positional head. Long distance → long-range / content-based head.

In [ ]:
def compute_attention_metrics(attn_matrix):
    """
    Computes per-head Shannon entropy (bits) and mean attention distance.
    attn_matrix: (1, n_heads, T, T) tensor with softmax probabilities.
    """
    _, n_heads, T, _ = attn_matrix.size()
    entropies, distances = [], []
    for h in range(n_heads):
        A = attn_matrix[0, h]  # (T, T)
        # Shannon entropy in BITS: use log2
        entropy = -(A * torch.log2(A + 1e-9)).sum(dim=-1).mean().item()
        entropies.append(entropy)
        # Mean attention distance
        idx = torch.arange(T, dtype=torch.float32, device=attn_matrix.device)
        dist = torch.abs(idx.unsqueeze(0) - idx.unsqueeze(1))
        mean_dist = (A * dist).sum(dim=-1).mean().item()
        distances.append(mean_dist)
    return entropies, distances


def plot_attention_heatmap(attn_matrix, layer_idx, head_idx, tokens,
                           figures_dir="figures", font_prop=None):
    """Saves attention heatmap. English labels use DejaVu Sans; Indic tick labels use font_prop."""
    _reset_to_latin()  # reset global font to Latin so title/labels render correctly
    os.makedirs(figures_dir, exist_ok=True)
    T = len(tokens)
    matrix = attn_matrix[0, head_idx, :T, :T].cpu().numpy()

    fig, ax = plt.subplots(figsize=(max(6, T * 0.55), max(5, T * 0.55)))
    sns.heatmap(
        matrix,
        ax=ax,
        cmap="viridis",
        square=True,
        cbar_kws={'label': 'Attention Weight'},
        xticklabels=tokens,
        yticklabels=tokens
    )
    # Apply explicit font properties to each tick label to guarantee Indic glyph rendering
    if font_prop is not None:
        for tick in ax.get_xticklabels():
            tick.set_fontproperties(font_prop)
            tick.set_fontsize(9)
        for tick in ax.get_yticklabels():
            tick.set_fontproperties(font_prop)
            tick.set_fontsize(9)
    ax.set_title(
        f"Attention Heatmap — Layer {layer_idx}, Head {head_idx} (TELUGU)",
        fontsize=11, fontfamily='DejaVu Sans'
    )
    ax.set_xlabel("Key Positions (Source)", fontsize=10, fontfamily='DejaVu Sans')
    ax.set_ylabel("Query Positions (Target)", fontsize=10, fontfamily='DejaVu Sans')
    plt.tight_layout()
    plot_path = os.path.join(figures_dir, f'attn_L{layer_idx}_H{head_idx}.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"[INFO] Saved {plot_path}")


In [ ]:
# ── Run attention analysis ──────────────────────────────────────────────────
figures_dir = "figures"
os.makedirs(figures_dir, exist_ok=True)

# Load font object for explicit tick-label rendering (fixes tofu rectangles in heatmap)
_font_prop = _indic_font_prop  # set in the imports cell

# Use a longer dedicated prompt for richer attention patterns
attn_prompt = 'భారతదేశం ఒక గొప్ప దేశం. ఇక్కడ అనేక భాషలు మాట్లాడతారు మరియు వివిధ సంస్కృతులు ఉన్నాయి.'
input_ids = torch.tensor(tokenizer.encode(attn_prompt).ids, dtype=torch.long).unsqueeze(0).to(device)
model.eval()
with torch.no_grad():
    _, _, all_attns = model(input_ids)

# Decode each token individually for axis labels (strip SentencePiece ▁ prefix for clean display)
token_ids  = tokenizer.encode(attn_prompt).ids
token_strs = [tokenizer.decode([tid]).replace('▁', '').strip() or '▪' for tid in token_ids]
n_layers   = len(all_attns)
n_heads    = all_attns[0].shape[1]

# Plot early layer (0) and late layer (-1), heads 0 / 1 / 2
for layer_idx in [0, n_layers - 1]:
    for head_idx in range(min(3, n_heads)):
        plot_attention_heatmap(
            all_attns[layer_idx], layer_idx=layer_idx,
            head_idx=head_idx, tokens=token_strs,
            figures_dir=figures_dir, font_prop=_font_prop
        )

# Compute entropy + distance for EVERY layer and head
all_layer_entropy  = []
all_layer_distance = []
for layer_idx, attn in enumerate(all_attns):
    ent, dist = compute_attention_metrics(attn)
    all_layer_entropy.append(ent)
    all_layer_distance.append(dist)
    print(f"Layer {layer_idx}:")
    for h in range(len(ent)):
        print(f"  Head {h} | Entropy: {ent[h]:.4f} bits | Mean Dist: {dist[h]:.2f} tokens")


In [ ]:
# ── Entropy and Distance bar-chart across all layers and heads ──────────────
def plot_attention_stats(all_layer_entropy, all_layer_distance, figures_dir="figures"):
    _reset_to_latin()  # reset global font before building bar chart
    os.makedirs(figures_dir, exist_ok=True)
    n_layers = len(all_layer_entropy)
    n_heads  = len(all_layer_entropy[0])
    x = np.arange(n_layers)
    width = 0.8 / n_heads

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for h in range(n_heads):
        ent_vals  = [all_layer_entropy[l][h]  for l in range(n_layers)]
        dist_vals = [all_layer_distance[l][h] for l in range(n_layers)]
        offset = (h - n_heads / 2 + 0.5) * width
        axes[0].bar(x + offset, ent_vals,  width=width * 0.9, label=f'Head {h}')
        axes[1].bar(x + offset, dist_vals, width=width * 0.9, label=f'Head {h}')

    axes[0].set_xticks(x)
    axes[0].set_xticklabels([f'L{i}' for i in range(n_layers)])
    axes[0].set_ylabel('Shannon Entropy (bits)')
    axes[0].set_title('Attention Entropy per Head/Layer (TELUGU)')
    axes[0].legend(fontsize=7)
    axes[0].grid(axis='y', linestyle='--', alpha=0.5)

    axes[1].set_xticks(x)
    axes[1].set_xticklabels([f'L{i}' for i in range(n_layers)])
    axes[1].set_ylabel('Mean Attention Distance (tokens)')
    axes[1].set_title('Mean Attention Distance per Head/Layer (TELUGU)')
    axes[1].legend(fontsize=7)
    axes[1].grid(axis='y', linestyle='--', alpha=0.5)

    # Force standard (non-Indic) font for all axis/legend text in the bar chart
    _latin_font = {'fontfamily': 'DejaVu Sans', 'fontsize': 9}
    for ax in axes:
        for lbl in ax.get_xticklabels() + ax.get_yticklabels():
            lbl.set_fontfamily('DejaVu Sans')
        ax.set_xlabel(ax.get_xlabel(), fontfamily='DejaVu Sans')
        ax.set_ylabel(ax.get_ylabel(), fontfamily='DejaVu Sans')
        ax.set_title(ax.get_title(), fontfamily='DejaVu Sans')
        legend = ax.get_legend()
        if legend:
            for text in legend.get_texts():
                text.set_fontfamily('DejaVu Sans')

    plt.tight_layout()
    path = os.path.join(figures_dir, 'attention_entropy_distance.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"[INFO] Attention stats chart saved to {path}")

plot_attention_stats(all_layer_entropy, all_layer_distance, figures_dir=figures_dir)


## 11. Qualitative Notes and Attention Interpretation

*(Fill in after inspecting the plots above)*

### Coherence and language correctness
- **Greedy (T=0):** Tends to produce deterministic but potentially repetitive output. Likely to fall into local repetition loops for a small pre-trained model.
- **T=0.5:** More conservative sampling; output is more focused but may still lack fluency.
- **T=1.0:** Standard sampling; a balance between diversity and coherence.
- **T=1.5:** High entropy sampling; generates more diverse but potentially incoherent text.

### Attention head interpretation
- **Low entropy + short mean distance** → positional / local head (attends to adjacent tokens).
- **Low entropy + long mean distance** → content-based / syntactic head (long-range dependencies).
- **High entropy** → diffuse head (uniform attention; may be a 'rest' head in early training).

Early layers (Layer 0) tend to have higher entropy (more diffuse) and shorter attention distance. Late layers often show sharper, longer-range attention patterns as they encode higher-level semantics.

*(Describe specific observations from your model's heatmaps here.)*